# CUDA-MEEP vs Meep: GPU Benchmark

**Strategy:**
- CPU results (CUDA-MEEP CPU + Meep) are pre-measured locally and stored in `benchmarks/cpu_results.json`
- This notebook measures **GPU** throughput on Colab T4, then combines both for the full comparison

**Local CPU results (already measured):**

| Grid | CUDA-MEEP CPU | Meep CPU |
|------|--------------|----------|
| 64²  | 1.5 Mcells/s | 19.5 Mcells/s |
| 128² | 6.1 Mcells/s | 20.6 Mcells/s |
| 256² | 8.8 Mcells/s | 21.6 Mcells/s |
| 512² | 15.4 Mcells/s | 19.2 Mcells/s |

> **Enable GPU first:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Check GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NOT DETECTED — enable GPU in Runtime settings')

In [ ]:
# Step 2: Clone repo
!git clone https://github.com/shahzaibshazoo/cuda-meep.git
!pip install torch numpy matplotlib --quiet

In [ ]:
# Step 3: Setup
import sys, time, json
import numpy as np
sys.path.insert(0, '/content/cuda-meep/src')
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Part 1: CUDA-MEEP GPU Benchmark

In [ ]:
from core import YeeGrid, FieldSet, MurABC, GaussianPulse, PointSource, SourceCollection, FDTD2D

GRID_SIZES = [64, 128, 256, 512]
N_WARMUP   = 20
N_STEPS    = 200
DX         = 1e-3
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_cuda_meep(N, device):
    grid     = YeeGrid(N, N, dx=DX, dy=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC(grid, fields.Hz)
    pulse    = GaussianPulse(amplitude=1.0, sigma=30*grid.dt)
    src      = PointSource(pulse, N//2, N//2, 'Hz', grid=grid, N_steps=N_WARMUP+N_STEPS)
    sim      = FDTD2D(grid, fields, boundary, SourceCollection([src]), n_check=500)
    sim.run(N_WARMUP)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    sim.run(N_STEPS)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return N_STEPS * N * N / elapsed / 1e6, elapsed / N_STEPS * 1000

gpu_results = []
print(f'--- CUDA-MEEP ({DEVICE.upper()}) ---')
for N in GRID_SIZES:
    m, ms = run_cuda_meep(N, DEVICE)
    gpu_results.append({'N': N, 'mcells_s': m, 'ms_step': ms})
    print(f'  {N:4d}²   {m:8.1f} Mcells/s   {ms:8.3f} ms/step')

## Part 2: Load CPU Results (pre-measured locally)

In [ ]:
# Load the JSON saved by running: python benchmarks/cpu_benchmark.py  (locally with pymeep)
cpu_json_path = '/content/cuda-meep/benchmarks/cpu_results.json'
with open(cpu_json_path) as f:
    cpu_data = json.load(f)

cuda_cpu   = cpu_data['cuda_meep_cpu']   # CUDA-MEEP on laptop CPU
meep_cpu   = cpu_data['meep_cpu']        # Meep on laptop CPU
meta       = cpu_data['meta']

print(f'CPU results from: {meta["date"]}')
print(f'Platform: {meta["platform"]}  |  PyTorch {meta["torch"]}')
print()
print('--- CUDA-MEEP (laptop CPU) ---')
for r in cuda_cpu:
    print(f'  {r["N"]:4d}²   {r["mcells_s"]:8.1f} Mcells/s   {r["ms_step"]:8.3f} ms/step')
print('--- Meep (laptop CPU) ---')
for r in meep_cpu:
    print(f'  {r["N"]:4d}²   {r["mcells_s"]:8.1f} Mcells/s   {r["ms_step"]:8.3f} ms/step')

## Part 3: Full Comparison Table

In [ ]:
gpu_name = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'

print('=' * 80)
print(f'  {"Grid":5s}  {"GPU (Mcells/s)":>16s}  {"CPU (Mcells/s)":>16s}  {"Meep (Mcells/s)":>16s}  {"GPU/Meep":>10s}')
print(f'  {"":5s}  {gpu_name[:16]:>16s}  {"CUDA-MEEP":>16s}  {"pymeep":>16s}  {"speedup":>10s}')
print('=' * 80)

for i, N in enumerate(GRID_SIZES):
    gpu  = gpu_results[i]['mcells_s']
    cpu  = cuda_cpu[i]['mcells_s']
    meep = meep_cpu[i]['mcells_s']
    speedup_vs_meep = gpu / meep
    speedup_vs_cpu  = gpu / cpu
    print(f'  {N}²    {gpu:>14.1f}  {cpu:>14.1f}  {meep:>14.1f}  {speedup_vs_meep:>8.1f}x')

print('=' * 80)
print()

# Average speedup at larger grids (256+ where GPU wins)
large_grid_idx = [i for i, N in enumerate(GRID_SIZES) if N >= 256]
avg_vs_meep = sum(gpu_results[i]['mcells_s']/meep_cpu[i]['mcells_s'] for i in large_grid_idx) / len(large_grid_idx)
avg_vs_cpu  = sum(gpu_results[i]['mcells_s']/cuda_cpu[i]['mcells_s'] for i in large_grid_idx) / len(large_grid_idx)

print(f'Average GPU speedup over Meep  (256²+): {avg_vs_meep:.1f}x')
print(f'Average GPU speedup over CPU   (256²+): {avg_vs_cpu:.1f}x')
print()

crossover = [N for i,N in enumerate(GRID_SIZES) if gpu_results[i]['mcells_s'] <= cuda_cpu[i]['mcells_s']]
if crossover:
    print(f'Note: GPU slower than CPU at {crossover} — kernel launch overhead dominates')
    print('      at small grids. GPU wins decisively at 256² and above.')

## Part 4: Speedup Chart

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'CUDA-MEEP vs Meep — GPU: {gpu_name}', fontsize=13, fontweight='bold')

# Left: Throughput
ax = axes[0]
ax.plot(GRID_SIZES, [r['mcells_s'] for r in gpu_results],  'o-g', lw=2, ms=9,
        label=f'CUDA-MEEP GPU ({gpu_name})')
ax.plot(GRID_SIZES, [r['mcells_s'] for r in cuda_cpu],     's-b', lw=2, ms=9,
        label='CUDA-MEEP CPU (laptop)')
ax.plot(GRID_SIZES, [r['mcells_s'] for r in meep_cpu],     '^-r', lw=2, ms=9,
        label='Meep CPU (laptop)')
ax.set(xlabel='Grid size', ylabel='Throughput (Mcells/s)',
       title='Throughput Comparison')
ax.set_xticks(GRID_SIZES); ax.set_xticklabels([f'{N}²' for N in GRID_SIZES])
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: GPU speedup over Meep
ax2 = axes[1]
speedups_meep = [gpu_results[i]['mcells_s']/meep_cpu[i]['mcells_s'] for i in range(len(GRID_SIZES))]
speedups_cpu  = [gpu_results[i]['mcells_s']/cuda_cpu[i]['mcells_s'] for i in range(len(GRID_SIZES))]
x = range(len(GRID_SIZES))
w = 0.35
b1 = ax2.bar([i-w/2 for i in x], speedups_meep, width=w,
             color=['red' if s<1 else '#2ca02c' for s in speedups_meep],
             alpha=0.85, label='vs Meep')
b2 = ax2.bar([i+w/2 for i in x], speedups_cpu, width=w,
             color=['red' if s<1 else 'steelblue' for s in speedups_cpu],
             alpha=0.85, label='vs CUDA-MEEP CPU')
ax2.bar_label(b1, fmt='%.1fx', fontsize=10, padding=2)
ax2.bar_label(b2, fmt='%.1fx', fontsize=10, padding=2)
ax2.axhline(1, color='black', ls='--', alpha=0.4)
ax2.set(ylabel='Speedup', title=f'GPU Speedup (GPU = {gpu_name})')
ax2.set_xticks(list(x)); ax2.set_xticklabels([f'{N}²' for N in GRID_SIZES])
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /content/benchmark_results.png')

## Part 5: Brain Tumor Detection Demo

In [ ]:
import os
os.chdir('/content/cuda-meep')
print('Running 16-antenna MIMO brain tumor detection (32 FDTD simulations)...')
print(f'Device: {DEVICE.upper()}  |  Grid: 150×150  |  N_steps: 1000')
r = subprocess.run([sys.executable, 'examples/brain_mimo_imaging.py'],
                   capture_output=True, text=True, timeout=1200)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr[-2000:])

In [ ]:
from IPython.display import Image, display
img = '/content/cuda-meep/examples/output/brain_mimo_imaging.png'
display(Image(filename=img)) if os.path.exists(img) else print('Image missing — check errors above')